In [8]:
import pandas as pd
import numpy as np
import os

# 核心配置区（读取上阶段保存本地的数据）
DATA_DIR = 'G:/quant_data/daily_bar/market_data'

print("reading...")
df_close  = pd.read_csv(os.path.join(DATA_DIR, 'close_matrix.csv'), index_col='Date', parse_dates=True)
df_open   = pd.read_csv(os.path.join(DATA_DIR, 'open_matrix.csv'), index_col='Date', parse_dates=True)
df_vol    = pd.read_csv(os.path.join(DATA_DIR, 'vol_matrix.csv'), index_col='Date', parse_dates=True)
df_amount = pd.read_csv(os.path.join(DATA_DIR, 'turnover_matrix.csv'), index_col='Date', parse_dates=True) # 昨晚用amount代指的特征

print("finish reading")

# 1. 计算收益率矩阵：pct_change函数计算数据百分比变化率
df_return = df_close.pct_change()

#2.Factor 1：5日滚动量价相关性矩阵 (Price-Volume Divergence)
#输入close和volume矩阵,.rolling(n)是在每一次计算前取n天数据
#和。corr一起计算5天收盘价和成交量的皮尔逊相关系数 
df_factor_pvd = df_close.rolling(5).corr(df_vol)

#3.Factor 2：非流动性因子矩阵 (Illiquidity Proxy)
# 公式：绝对收益率 / (成交额 / 1,000,000) 
# 成交额除以100万是为了防止分母数值过大导致因子值变成极小的科学计数法
#.replace(0, np.nan)防止分母为零情况发生
df_factor_illiq = df_return.abs() / (df_amount / 1000000).replace(0, np.nan)

print("Factor count finished")

#数据清洗：横截面MAD去极值与中位数填充
def clean_factor_cross_section(df_factor):
    """
    对输入的二维时序矩阵进行逐日（横截面 Axis=1）的 MAD 去极值和缺失值填充
    """
    cleaned_df = df_factor.copy()
    
    # 逐日进行横截面处理
    for date in df_factor.index:
        row = df_factor.loc[date]
        if row.isnull().all():
            continue
            
        # 1.计算当前这一天，全市场个股的中位数
        median = row.median()
        # 2.计算绝对偏离值的中位数 (MAD)
        mad = (row - median).abs().median()
        
        # 防止整个横截面数据完全一致导致 mad 为 0 的边界崩塌
        if mad == 0:
            mad = 1e-6
            
        # 3.算上下边界 (3倍MAD)
        floor = median - 3 * mad
        caps = median + 3 * mad
        
        # 4.裁剪极值并填补缺失值（用当天最安全的横截面中位数填补停牌股的 NaN）
        row_clipped = np.clip(row, floor, caps)
        row_filled = row_clipped.fillna(median)
        
        cleaned_df.loc[date] = row_filled
        
    return cleaned_df

df_pvd_clean   = clean_factor_cross_section(df_factor_pvd)
df_illiq_clean = clean_factor_cross_section(df_factor_illiq)

#因子保存
df_pvd_clean.to_csv(os.path.join(DATA_DIR, 'factor_pvd_clean.csv'))
df_illiq_clean.to_csv(os.path.join(DATA_DIR, 'factor_illiq_clean.csv'))

print("Factor clearn finished")
print(df_pvd_clean.tail())
print(f"\n saving finished，size: {df_pvd_clean.shape}")

reading...
finish reading
Factor count finished
Factor clearn finished
              000001    000002    600000    600016    600028    600030  \
Date                                                                     
2026-06-05  0.133264  0.965181  0.920610  0.101890  0.393716  0.426223   
2026-06-08  0.286805  0.931238  0.942098  0.141241  0.209562 -0.332372   
2026-06-09  0.749759  0.875800  0.840701  0.600606  0.570686 -0.023889   
2026-06-10  0.973270  0.734985  0.977127  0.934070  0.732681  0.074325   
2026-06-11  0.776971  0.764492  0.628910  0.354443  0.775072  0.139282   

              600036    600050    600104    600519  
Date                                                
2026-06-05  0.642208 -0.175881  0.055213  0.295528  
2026-06-08  0.743128 -0.711005 -0.629636  0.317427  
2026-06-09  0.219668 -0.028414 -0.253686  0.855046  
2026-06-10  0.706656  0.457615  0.074325  0.828448  
2026-06-11  0.472604  0.421043 -0.360345  0.216252  

 saving finished，size: (104, 10)


In [3]:
import tushare as ts
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import os
import time

# 核心配置区：数据位和token补充
MY_TOKEN = '记得换上自己token'
DATA_DIR = 'G:/quant_data/daily_bar/market_data'

# Clean up proxy environment to avoid potential connection dropping（开着代理运行不出错的保障代码）
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''

ts.set_token(MY_TOKEN)
pro = ts.pro_api()

# 1.读入你目前本地已经算好的原始因子
df_illiq = pd.read_csv(os.path.join(DATA_DIR, 'factor_illiq_clean.csv'), index_col='Date', parse_dates=True)

trade_days = df_illiq.index.strftime('%Y%m%d').tolist()
pure_codes = df_illiq.columns.tolist()
ts_code_list = [f"{code}.SZ" if code.startswith('00') else f"{code}.SH" for code in pure_codes]

#第一步：云端拉取这10只股年初至今的每日总市值，构建市值矩阵
print("正在获取这10只股票的每日总市值以构建控制矩阵")
all_mv_data = []

for date in trade_days:
    try:
        # daily_basic接口可以一发拿到当天全市场的市值数据
        df_basic = pro.daily_basic(trade_date=date, fields='ts_code,trade_date,total_mv')
        if df_basic is not None and not df_basic.empty:
            df_filter = df_basic[df_basic['ts_code'].isin(ts_code_list)].copy()
            all_mv_data.append(df_filter)
        time.sleep(0.1) #呼吸延迟
    except Exception as e:
        print(f"日期 {date} 市值获取失败: {e}")

# 组装成二维矩阵
df_total_mv = pd.concat(all_mv_data, ignore_index=True)
df_total_mv['pure_code'] = df_total_mv['ts_code'].apply(lambda x: x.split('.')[0])
df_total_mv['trade_date'] = pd.to_datetime(df_total_mv['trade_date'], format='%Y%m%d')
mv_matrix = df_total_mv.pivot_table(index='trade_date', columns='pure_code', values='total_mv').sort_index()

# 核心：将市值取自然对数 ln(Size)
ln_mv_matrix = np.log(mv_matrix)

print("本地对数市值矩阵构建成功")

# 回归跑残差
print("\n开始执行横截面OLS线性回归中性化")

# 1. 显式开启 Pandas 的未来新特性，避免Downcasting告警
pd.set_option('future.no_silent_downcasting', True)

# 建立一个空矩阵，用来装洗干净的纯净因子残差
df_illiq_neutral = pd.DataFrame(index=df_illiq.index, columns=df_illiq.columns)

for date in df_illiq.index:
    y = df_illiq.loc[date]    # 当天10只股的原始因子值 (Y)
    x = ln_mv_matrix.loc[date] # 当天10只股的对数市值 (X)
    
    # 确保没有空值干扰回归
    valid_idx = y.notnull() & x.notnull()
    if valid_idx.sum() < 3: # 样本太少则无法拟合回归线
        continue
        
    Y = y[valid_idx].values
    X = x[valid_idx].values
    X_with_constant = sm.add_constant(X) # 强行加上截距项 alpha
    
    # 最小二乘法回归拟合
    model = sm.OLS(Y, X_with_constant)
    results = model.fit()
    
    # 提取残差 (Residuals)
    df_illiq_neutral.loc[date, valid_idx] = results.resid

# 3. 横截面中位数填充
# 先用 infer_objects 转换类型，再安全填充，告警彻底消失
df_illiq_neutral = df_illiq_neutral.infer_objects(copy=False)
df_illiq_neutral = df_illiq_neutral.apply(lambda row: row.fillna(row.median() if row.notnull().any() else 0.0), axis=1).astype(float)

# 第三步：纯净 Alpha 特征
df_illiq_neutral.to_csv(os.path.join(DATA_DIR, 'factor_illiq_neutral.csv'))
print("\n 纯净因子矩阵构建成功")

正在获取这10只股票的每日总市值以构建控制矩阵
本地对数市值矩阵构建成功

开始执行横截面OLS线性回归中性化

 纯净因子矩阵构建成功
